In [ ]:
import torch
from datasets import load_dataset
from transformers import(
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import pandas as pd
import numpy as np
import evaluate

c:\Users\prodk\anaconda3\envs\D.L_inflearn\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = load_dataset('sepidmnorozy/Korean_sentiment')
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 36000
    })
    validation: Dataset({
        features: ['label', 'text'],
        num_rows: 1333
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 2667
    })
})

In [ ]:
print(dataset['train'][3118])
print(dataset['train'][14310])

{'label': 1, 'text': '졸잼!!!성아가나중에억울한일이잇어서좀슬펏는데마지막은기쁘게끝나서다행이에여'}
{'label': 0, 'text': '진짜 어떻게 된놈의 영화가 여고괴담 1보다도 못함? 신기하다 그것도 2012년작이 1998년보다 못함 솔까 여고괴담1은 반전은 최고지 뭐 이놈의 영화는 여고괴담 시리즈보다도 못하는거같다'}


# 토큰화 Tokenize

In [ ]:
model_name = 'kykim/bert-kor-base'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer

BertTokenizerFast(name_or_path='kykim/bert-kor-base', vocab_size=42000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [ ]:
def tokenizer_func(x):
    return tokenizer(
        x['text'],
        padding='max_length',
        max_length=256,
        truncation=True
    )

In [ ]:
tokenized_datasets = dataset.map(tokenizer_func, batched=True)

Map: 100%|██████████| 2667/2667 [00:00<00:00, 5577.77 examples/s]


In [ ]:
train_num_samples = 10000

train_ds = tokenized_datasets['train'].shuffle(seed=42).select(range(train_num_samples))
eval_ds = tokenized_datasets['validation'].shuffle(seed=42)

# 전이학습 Transfer Learning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at kykim/bert-kor-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# HyperParams

In [ ]:
bs = 32
epochs = 4
lr = 1e-5

In [ ]:
args = TrainingArguments(
    'outputs',
    learning_rate = lr,
    warmup_ratio = 0.1,
    lr_scheduler_type = 'cosine',
    fp16 = True,
    evaluation_strategy='epoch',
    per_device_train_batch_size = bs,
    per_device_eval_batch_size = bs,
    gradient_accumulation_steps=4, #until bs =128
    eval_accumulation_steps = 4,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to = 'none'
)

c:\Users\prodk\anaconda3\envs\D.L_inflearn\lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# Metrics

In [ ]:
metric = evaluate.load('accuracy')

# all Transformers models retrun logits
def compute_metrics(eval_pred):
    logits, labels =eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
class CustomTrainer(Trainer):
    def _save_checkpoint(self, model, trial, metrics=None):
        # 모델을 저장하기 전에 모든 텐서를 contiguous로 만듦
        for name, param in model.named_parameters():
            if param is not None:
                param.data = param.data.contiguous()
                if param.grad is not None:
                    param.grad.data = param.grad.data.contiguous()
        super()._save_checkpoint(model, trial, metrics)

# Trainer

In [ ]:
trainer = CustomTrainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

c:\Users\prodk\anaconda3\envs\D.L_inflearn\lib\site-packages\accelerate\accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
trainer.train()

 25%|██▌       | 78/312 [07:24<03:17,  1.18it/s]   






















                                                
                                              
 25%|██▌       | 78/312 [07:28<03:17,  1.18it/s]


{'eval_loss': 0.31869375705718994, 'eval_accuracy': 0.8874718679669917, 'eval_runtime': 3.315, 'eval_samples_per_second': 402.116, 'eval_steps_per_second': 12.67, 'epoch': 1.0}


 50%|█████     | 156/312 [10:08<06:36,  2.54s/it]

























                                                 
                                                

 50%|█████     | 156/312 [10:13<06:36,  2.54s/it]



{'eval_loss': 0.3497461974620819, 'eval_accuracy': 0.8904726181545386, 'eval_runtime': 4.3465, 'eval_samples_per_second': 306.683, 'eval_steps_per_second': 9.663, 'epoch': 1.99}


 75%|███████▌  | 234/312 [15:01<01:13,  1.07it/s]  





















                                                 
                                                

 75%|███████▌  | 234/312 [15:08<01:13,  1.07it/s]



{'eval_loss': 0.3507227599620819, 'eval_accuracy': 0.8979744936234059, 'eval_runtime': 6.0489, 'eval_samples_per_second': 220.37, 'eval_steps_per_second': 6.943, 'epoch': 2.99}


100%|██████████| 312/312 [16:35<00:00,  1.11s/it]




















                                                 
                                                

100%|██████████| 312/312 [16:40<00:00,  1.11s/it]

                                                 
100%|██████████| 312/312 [16:40<00:00,  3.21s/it]  

{'eval_loss': 0.3554419279098511, 'eval_accuracy': 0.8957239309827457, 'eval_runtime': 3.333, 'eval_samples_per_second': 399.941, 'eval_steps_per_second': 12.601, 'epoch': 3.99}
{'train_runtime': 1000.1826, 'train_samples_per_second': 39.993, 'train_steps_per_second': 0.312, 'train_loss': 0.11484177907307942, 'epoch': 3.99}


TrainOutput(global_step=312, training_loss=0.11484177907307942, metrics={'train_runtime': 1000.1826, 'train_samples_per_second': 39.993, 'train_steps_per_second': 0.312, 'total_flos': 5247486888099840.0, 'train_loss': 0.11484177907307942, 'epoch': 3.987220447284345})

In [ ]:
trainer.save_model('./mymodels')

# 추론 Inference

In [ ]:
pipe = pipeline('text-classification', model='./mymodels')

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


# 테스트셋 사용

In [ ]:
test_data = dataset['validation'].shuffle(seed=424)[:100]
td = pd.DataFrame(test_data)
td

,label,text
0,1,"비밀은 적정선에 관한 이야기이기도 하다. 사람에게 돈이란, 평판이란, 책임이란, 기..."
1,0,메인 시나리오 뼈대가 없고 보여주기라기엔 용과의 전투씬도 너무 부족함..더욱이 결말...
2,1,올??? 찌질한 스커드가 노만 리더스 행님이었다니 !! ㅋㅋ
3,0,5점이 적당하나 평점이 쓸데없이 높아서 낮출 필요가 있슴...
4,0,"연출, 기획, 스토리 전부다 쓰레기..제인생 영화중에 제일 쓰레기.. 절대보지마요진심"
...,...,...
95,1,"""""""철아 엄마가 꼭 니 신세 갚고 죽을께"""""""
96,0,이건 뭐 .. 완전 삼류네
97,0,으... 밑도 끝도 없음. 여자가 처음부터 자기혈청을 제공했으면 카오스가 안왔을거 아님?
98,0,요즘에 보기에는 상당히 지루한 진부한 스릴러. 원초적 본능을 뛰어 넘기에도 힘든 영...


In [ ]:
preds = pipe(td['text'].tolist())

preds_df = pd.DataFrame(preds)
preds_df

,label,score
0,LABEL_1,0.997155
1,LABEL_0,0.998864
2,LABEL_0,0.953251
3,LABEL_1,0.924685
4,LABEL_0,0.998925
...,...,...
95,LABEL_1,0.988680
96,LABEL_0,0.998678
97,LABEL_0,0.995881
98,LABEL_0,0.999176


In [ ]:
preds_df.rename(columns={'label':'pred'}, inplace=True)
preds_df['pred'] = preds_df['pred'].map({'LABEL_1': 1, 'LABEL_0':0})

preds_df = pd.concat([preds_df, td], axis=1)

In [ ]:
preds_df

,pred,score,label,text
0,1,0.997155,1,"비밀은 적정선에 관한 이야기이기도 하다. 사람에게 돈이란, 평판이란, 책임이란, 기..."
1,0,0.998864,0,메인 시나리오 뼈대가 없고 보여주기라기엔 용과의 전투씬도 너무 부족함..더욱이 결말...
2,0,0.953251,1,올??? 찌질한 스커드가 노만 리더스 행님이었다니 !! ㅋㅋ
3,1,0.924685,0,5점이 적당하나 평점이 쓸데없이 높아서 낮출 필요가 있슴...
4,0,0.998925,0,"연출, 기획, 스토리 전부다 쓰레기..제인생 영화중에 제일 쓰레기.. 절대보지마요진심"
...,...,...,...,...
95,1,0.988680,1,"""""""철아 엄마가 꼭 니 신세 갚고 죽을께"""""""
96,0,0.998678,0,이건 뭐 .. 완전 삼류네
97,0,0.995881,0,으... 밑도 끝도 없음. 여자가 처음부터 자기혈청을 제공했으면 카오스가 안왔을거 아님?
98,0,0.999176,0,요즘에 보기에는 상당히 지루한 진부한 스릴러. 원초적 본능을 뛰어 넘기에도 힘든 영...


In [ ]:
mask = preds_df['pred'] == preds_df['label']
len(preds_df[mask])

90

In [ ]:
path = 'datas\mist_review.csv'
raw = pd.read_csv(path)
df = raw.copy()

In [ ]:
df_review = df['review_list']

In [ ]:
txts_td = pd.DataFrame(df_review)

In [ ]:
txts_td.head()

,review_list
0,피부 건조하거나 열 받았을 때 하면 꾸준하게 사용 해주고 있어요
1,피부 관리 하려고 계속 쓰고 있는 제품 입니다 n 부드러운 피부 기분 좋아지네요
2,믿고 쓰는 달바 미스트 n 유일하게 아무 때 나 뿌려도 건조해지지 않는 미스트
3,완전 좋아요 여름 뿌리 면 시원해서 자극 없고 흡수 잘 돼요
4,뿌리 고 나면 흡수 도 잘 되서 산뜻해서 좋아요 저녁 앰플 바르기 전 한번 뿌려주고있어요


In [ ]:
txts_td.to_csv('datas\데이터타입검사용(temp).csv')

In [ ]:
txts_td.dropna(inplace=True)

In [ ]:
txts_td.isnull().sum()

review_list    0
dtype: int64

In [ ]:
txts_td['review_list'] = txts_td['review_list'].astype(str)

In [ ]:
txts_td['review_list']

0                     피부 건조하거나 열 받았을 때 하면 꾸준하게 사용 해주고 있어요
1            피부 관리 하려고 계속 쓰고 있는 제품 입니다 n 부드러운 피부 기분 좋아지네요
2             믿고 쓰는 달바 미스트 n 유일하게 아무 때 나 뿌려도 건조해지지 않는 미스트
3                       완전 좋아요 여름 뿌리 면 시원해서 자극 없고 흡수 잘 돼요
4       뿌리 고 나면 흡수 도 잘 되서 산뜻해서 좋아요 저녁 앰플 바르기 전 한번 뿌려주고있어요
                              ...                        
3970              가을 겨울철 필수 템 오일 별로 안 좋아했는데 트릴로지 꼭 다시 사 요
3971    오일 내 정말 여러 개 써봣 지만 추잡하게 번 거리 지 않고 향 도 약간 오이 팩 ...
3972                        즈 힙 오일 사랑 입니다 피부 넘나 좋은 오일 이에요
3973                  처음 써 보는건데 촉촉히고 향 도 좋고 괜찮네요 ㅎㅎ 만족합니다
3974                      원래 쓰던 템 쿠팡 사려다가 가품 걱정 되어서 올 영 삼
Name: review_list, Length: 3884, dtype: object

In [ ]:
# 정규화
import re
txts_td['review_list'].apply(lambda x : re.sub(r'[^ㄱ-ㅣ가-힣]+', ' ', x))

0                     피부 건조하거나 열 받았을 때 하면 꾸준하게 사용 해주고 있어요
1              피부 관리 하려고 계속 쓰고 있는 제품 입니다 부드러운 피부 기분 좋아지네요
2               믿고 쓰는 달바 미스트 유일하게 아무 때 나 뿌려도 건조해지지 않는 미스트
3                       완전 좋아요 여름 뿌리 면 시원해서 자극 없고 흡수 잘 돼요
4       뿌리 고 나면 흡수 도 잘 되서 산뜻해서 좋아요 저녁 앰플 바르기 전 한번 뿌려주고있어요
                              ...                        
3970              가을 겨울철 필수 템 오일 별로 안 좋아했는데 트릴로지 꼭 다시 사 요
3971    오일 내 정말 여러 개 써봣 지만 추잡하게 번 거리 지 않고 향 도 약간 오이 팩 ...
3972                        즈 힙 오일 사랑 입니다 피부 넘나 좋은 오일 이에요
3973                  처음 써 보는건데 촉촉히고 향 도 좋고 괜찮네요 ㅎㅎ 만족합니다
3974                      원래 쓰던 템 쿠팡 사려다가 가품 걱정 되어서 올 영 삼
Name: review_list, Length: 3884, dtype: object

In [ ]:
txts_td['review_list']

0                     피부 건조하거나 열 받았을 때 하면 꾸준하게 사용 해주고 있어요
1            피부 관리 하려고 계속 쓰고 있는 제품 입니다 n 부드러운 피부 기분 좋아지네요
2             믿고 쓰는 달바 미스트 n 유일하게 아무 때 나 뿌려도 건조해지지 않는 미스트
3                       완전 좋아요 여름 뿌리 면 시원해서 자극 없고 흡수 잘 돼요
4       뿌리 고 나면 흡수 도 잘 되서 산뜻해서 좋아요 저녁 앰플 바르기 전 한번 뿌려주고있어요
                              ...                        
3970              가을 겨울철 필수 템 오일 별로 안 좋아했는데 트릴로지 꼭 다시 사 요
3971    오일 내 정말 여러 개 써봣 지만 추잡하게 번 거리 지 않고 향 도 약간 오이 팩 ...
3972                        즈 힙 오일 사랑 입니다 피부 넘나 좋은 오일 이에요
3973                  처음 써 보는건데 촉촉히고 향 도 좋고 괜찮네요 ㅎㅎ 만족합니다
3974                      원래 쓰던 템 쿠팡 사려다가 가품 걱정 되어서 올 영 삼
Name: review_list, Length: 3884, dtype: object

In [ ]:
# 데이터프레임을 데이터셋으로 변환
from datasets import Dataset

dataset = Dataset.from_pandas(txts_td)

In [ ]:
dataset

Dataset({
    features: ['review_list', '__index_level_0__'],
    num_rows: 3884
})

In [ ]:
# 모델 이름과 토크나이저 초기화
model_name = 'kykim/bert-kor-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# 토크나이저 함수 정의
def tokenizer_func(examples):
    return tokenizer(
        examples['review_list'],
        padding='max_length',
        max_length=256,
        truncation=True
    )

In [ ]:
tokenized_review_list = dataset.map(tokenizer_func, batched=True)

Map: 100%|██████████| 3884/3884 [00:00<00:00, 7585.39 examples/s]


In [ ]:
# 데이터셋 크기 확인
dataset_size = len(tokenized_review_list)
train_num_samples = min(10000, dataset_size)  # 데이터셋 크기를 넘지 않도록 설정

In [ ]:
txts_td = tokenized_review_list.select(range(train_num_samples))

In [ ]:
txts_td

Dataset({
    features: ['review_list', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3884
})

In [ ]:
preds_txts = pipe(txts_td['review_list'])

In [ ]:
preds_txts_df = pd.DataFrame(preds_txts)

In [ ]:
preds_txts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3884 entries, 0 to 3883
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   label   3884 non-null   object 
 1   score   3884 non-null   float64
dtypes: float64(1), object(1)
memory usage: 60.8+ KB


In [ ]:
preds_txts_df.head()

,label,score
0,LABEL_1,0.993719
1,LABEL_1,0.988343
2,LABEL_1,0.991620
3,LABEL_1,0.978445
4,LABEL_1,0.975487


In [ ]:
preds_txts_df['label'] = preds_txts_df['label'].map({'LABEL_1':1, 'LABEL_0':0})

In [ ]:
review_labeling_df = pd.DataFrame(txts_td)

In [ ]:
review_labeling_df.head()

,review_list,__index_level_0__,input_ids,token_type_ids,attention_mask
0,피부 건조하거나 열 받았을 때 하면 꾸준하게 사용 해주고 있어요,0,"[2, 14081, 14542, 15241, 5649, 35888, 3463, 14...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, ..."
1,피부 관리 하려고 계속 쓰고 있는 제품 입니다 n 부드러운 피부 기분 좋아지네요,1,"[2, 14081, 14549, 19012, 14238, 14616, 13979, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,믿고 쓰는 달바 미스트 n 유일하게 아무 때 나 뿌려도 건조해지지 않는 미스트,2,"[2, 15264, 14865, 3118, 8198, 19448, 2054, 217...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,완전 좋아요 여름 뿌리 면 시원해서 자극 없고 흡수 잘 돼요,3,"[2, 14289, 14105, 14347, 19032, 4134, 14634, 1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ..."
4,뿌리 고 나면 흡수 도 잘 되서 산뜻해서 좋아요 저녁 앰플 바르기 전 한번 뿌려주고있어요,4,"[2, 19032, 2260, 16562, 15421, 3238, 5957, 191...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [ ]:
# review_labeling_df.drop(columns=['__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'], inplace=True)

In [ ]:
preds_txts_df = pd.concat([preds_txts_df, review_labeling_df], axis=1)
preds_txts_df

,label,score,review_list
0,1,0.993719,피부 건조하거나 열 받았을 때 하면 꾸준하게 사용 해주고 있어요
1,1,0.988343,피부 관리 하려고 계속 쓰고 있는 제품 입니다 n 부드러운 피부 기분 좋아지네요
2,1,0.991620,믿고 쓰는 달바 미스트 n 유일하게 아무 때 나 뿌려도 건조해지지 않는 미스트
3,1,0.978445,완전 좋아요 여름 뿌리 면 시원해서 자극 없고 흡수 잘 돼요
4,1,0.975487,뿌리 고 나면 흡수 도 잘 되서 산뜻해서 좋아요 저녁 앰플 바르기 전 한번 뿌려주고있어요
...,...,...,...
3879,1,0.995162,가을 겨울철 필수 템 오일 별로 안 좋아했는데 트릴로지 꼭 다시 사 요
3880,1,0.973326,오일 내 정말 여러 개 써봣 지만 추잡하게 번 거리 지 않고 향 도 약간 오이 팩 ...
3881,1,0.993903,즈 힙 오일 사랑 입니다 피부 넘나 좋은 오일 이에요
3882,1,0.977997,처음 써 보는건데 촉촉히고 향 도 좋고 괜찮네요 ㅎㅎ 만족합니다


In [ ]:
preds_txts_df.to_csv('datas/mist_review_labeling.csv')

In [ ]:
path = './/datas/mist_review_labeling.csv'
df = pd.read_csv(path)

In [ ]:

# 같은 행에서 df['label']의 값이 1이면 df['review_list']에 [긍정]이라는 텍스트를 추가
df.loc[df['label'] == 1, 'review_list'] = '[긍정] ' + df['review_list']

# 같은 행에서 df['label']의 값이 0이면 df['review_list']에 [부정]이라는 텍스트를 추가
df.loc[df['label'] == 0, 'review_list'] = '[부정] ' + df['review_list']

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3884 entries, 0 to 3883
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   label        3884 non-null   int64  
 1   score        3884 non-null   float64
 2   review_list  3884 non-null   object 
dtypes: float64(1), int64(1), object(1)
memory usage: 91.2+ KB


In [ ]:
# df.drop(columns='Unnamed: 0', inplace=True)

In [ ]:
df.reset_index(drop=True)

,label,score,review_list
0,1,0.993719,[긍정] 피부 건조하거나 열 받았을 때 하면 꾸준하게 사용 해주고 있어요
1,1,0.988343,[긍정] 피부 관리 하려고 계속 쓰고 있는 제품 입니다 n 부드러운 피부 기분 좋아지네요
2,1,0.991620,[긍정] 믿고 쓰는 달바 미스트 n 유일하게 아무 때 나 뿌려도 건조해지지 않는 미스트
3,1,0.978445,[긍정] 완전 좋아요 여름 뿌리 면 시원해서 자극 없고 흡수 잘 돼요
4,1,0.975487,[긍정] 뿌리 고 나면 흡수 도 잘 되서 산뜻해서 좋아요 저녁 앰플 바르기 전 한번...
...,...,...,...
3879,1,0.995162,[긍정] 가을 겨울철 필수 템 오일 별로 안 좋아했는데 트릴로지 꼭 다시 사 요
3880,1,0.973326,[긍정] 오일 내 정말 여러 개 써봣 지만 추잡하게 번 거리 지 않고 향 도 약간 ...
3881,1,0.993903,[긍정] 즈 힙 오일 사랑 입니다 피부 넘나 좋은 오일 이에요
3882,1,0.977997,[긍정] 처음 써 보는건데 촉촉히고 향 도 좋고 괜찮네요 ㅎㅎ 만족합니다


In [ ]:
df.to_csv('datas/mist_review_labeling.csv')